# Data Analysis

## Context
For the module "Infographics" I'm trying to analyse data regarding the time usage of people aged 15 - 29 year old, reflecting the typical age range at CODE University of Applied Sciences. 

## Problem question
How do teenagers spent their time on a daily basis? And how did that change from 1991 to 2001 to 2012 to 2022? And what can be derived to the impact of smartphones and social media?

## Data
Forschungsdatenzentren Der Statistischen Ämter Des Bundes Und Der Länder. (2018a). Zeitbudgeterhebung 1991/1992, PUF, Version 0 (Version 0) [Data set]. Research Data Centres of the Statistical Offices of the Federation and the Federal States. https://doi.org/10.21242/63911.1992.00.00.4.1.0

Forschungsdatenzentren Der Statistischen Ämter Des Bundes Und Der Länder. (2018b). Zeitbudgeterhebung 2001/2002, PUF, Version 0 (Version 0) [Data set]. Research Data Centres of the Statistical Offices of the Federation and the Federal States. https://doi.org/10.21242/63911.2002.00.00.4.1.0

Forschungsdatenzentren Der Statistischen Ämter Des Bundes Und Der Länder. (2018c). Zeitverwendungserhebung 2012/2013, PUF, Version 0 (Version 0) [Data set]. Research Data Centres of the Statistical Offices of the Federation and the Federal States. https://doi.org/10.21242/63911.2013.00.00.4.1.0

Forschungsdatenzentren Der Statistischen Ämter Des Bundes Und Der Länder. (2018d). Zeitverwendungserhebung 2022, PUF, Version 0 (Version 0) [Data set]. Research Data Centres of the Statistical Offices of the Federation and the Federal States. https://doi.org/10.21242/63911.2022.00.00.4.1.1

## Deliverables
The aim of this data analysis is to find patterns and trends in how the spent time changed from 1991 to 2001, to 2012 and to finally 2022. To achieve that, the data analysis should provide as a first deliverable the following **data table**:
- activity_id
- activity_label
- year
- people
- mean
- median
- std
- mean+2std (as of 2001)
- mean+2std_pct

As supporting material the data analysis should provide these additional deliverables:
- ...

## Approach
The data analysis is based on the learned methods and material of "Data Science Basics".

### Notebook Setup

In [1]:
# importing the required modules for this project

import pyreadstat
import polars as pl
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt

### Data Import
As a first step I will load the available data sets to dataframe to process them further in this notebook.
As a first filter I will adjust the dataframe to the age group 15-29 year olds or in the case of 1991, the age groups 5-7.

#### Persons
The persons of the time spent survey are collected in a separate file, I need to retrieve the IDs that suit my desired age group. 

In [ ]:
## Create a new dataframe "YYYY_AG1629" from ...pers... and fill with filtered list, filter set to alterx=16-29

## for 2001 - 2022, the age groups are:
#   ...
#   7 = 7 bis 9 Jahre
#   10-74 = Jahre
#   75 = 75 bis 79 Jahre
#   ...

## for 1991 the age groups are:
# ...
# 4 = 12-15
# 5 = 15-20
# 6 = 20-25
# 7 = 25-30
# 8 = 30-35
# ...

## 1991
df_1991_pandas, metadata = pyreadstat.read_por("1991/zbe_ps.por")
df_1991 = pl.from_pandas(df_1991_pandas)
df_1991_AG1629 = df_1991.filter(
    pl.col('E51X').is_between(5,7))

## 2001
df_2001 = pl.read_csv('2001/zbe01_02puf_p.csv', separator=",", ignore_errors=True)
df_2001_AG1629 = df_2001.filter(
    pl.col('PH01B2X').is_between(15,29))

## 2012
df_2012 = pl.read_csv('2012/zve13_puf_pers.csv', separator=";", ignore_errors=True)
df_2012_AG1629 = df_2012.filter(
    pl.col('alterx').is_between(15,29))

## 2022
df_2022 = pl.read_csv('2022/zve_pers_puf_2022_v1.csv', separator=";", ignore_errors=True)
df_2022_AG1629 = df_2022.filter(
    pl.col('alterx').is_between(15,29))

#### Time entries
I'm importing the data for the time entries and filtering them by the retrieved relevant person IDs.

In [3]:
## Create a new dataframe "YYYY_AG16-29_activites" from ...sum... with only rows according to listed id_persx of "YYYY_AG16-29"

## 1991
# Using household/person pairs belonging to ages 16–29
pers_ids_1991_AG1629 = df_1991_AG1629.select(
    ["IDHH", "ID1"]
).unique()

# Load the diary records
df_1991_activities_pandas, metadata = pyreadstat.read_por("1991/zbe_sum.por")
df_1991_activities = pl.from_pandas(df_1991_activities_pandas)

# Keep diary records belonging to those people
df_1991_AG1629_activities = df_1991_activities.join(
    pers_ids_1991_AG1629,
    on=["IDHH", "ID1"],
    how="semi",
)

## 2001
# Using household/person pairs belonging to ages 16–29
pers_ids_2001_AG1629 = df_2001_AG1629.select(
    ["IDHH", "IDPERS"]
).unique()

# Load the diary records
df_2001_activities = pl.read_csv(
    "2001/zbe01_02puf_sum.csv",
    separator=",",
    ignore_errors=True,
)

# Keep diary records belonging to those people
df_2001_AG1629_activities = df_2001_activities.join(
    pers_ids_2001_AG1629,
    on=["IDHH", "IDPERS"],
    how="semi",
)

## 2012
# Using household/person pairs belonging to ages 16–29
pers_ids_2012_AG1629 = df_2012_AG1629.select(
    ["id_persx", "id_hhx"]
).unique()

# Load the diary records
df_2012_activities = pl.read_csv(
    "2012/zve13_puf_sum.csv",
    separator=";",
    ignore_errors=True,
)

# Keep diary records belonging to those people
df_2012_AG1629_activities = df_2012_activities.join(
    pers_ids_2012_AG1629,
    on=["id_persx", "id_hhx"],
    how="semi",
)

## 2022
# Using household/person pairs belonging to ages 16–29
pers_ids_2022_AG1629 = df_2022_AG1629.select(
    ["id_persx", "id_hhx"]
).unique()

# Load the diary records
df_2022_activities = pl.read_csv(
    "2022/zve_sum_puf_2022_v1.csv",
    separator=";",
    ignore_errors=True,
)

# Keep diary records belonging to those people
df_2022_AG1629_activities = df_2022_activities.join(
    pers_ids_2022_AG1629,
    on=["id_persx", "id_hhx"],
    how="semi",
)

_**Cleaning the house**_
I only want to work with the activity columns going forward

In [4]:
## Trim the dataframes to only the activity columns
df_1991_AG1629_activities = df_1991_AG1629_activities.select(
    'M12',
    pl.col(r'^ZH.*$')
).select(
    pl.exclude('ZHOEFF','ZHFAM','ZHPERS')
)

df_2001_AG1629_activities = df_2001_AG1629_activities.select(
    'GPZV80',
    pl.col(r'^ZH.*$')
)

df_2012_AG1629_activities = df_2012_AG1629_activities.select(
    'hrft80',
    pl.col(r'^ha_.*$')
)

df_2022_AG1629_activities = df_2022_AG1629_activities.select(
    'hrf_t80',
    pl.col(r'^ha_.*$')
)

_**Cleaning the house**_ Unfortunately the data uses different values for the time entries. I'll harmonise the data sets to minutes.

In [5]:
## Harmonise the time values to minutes

# 1991 uses minutes (f64), 2001 uses minutes (i64), 2012 uses seconds (i64), 2022 uses hh:mm (str)

# 1991
# already in minutes

# 2001
# already in minutes

# 2012 
# divide all values by 60 to go to minutes
df_2012_AG1629_activities = df_2012_AG1629_activities.with_columns(
        pl.col("^ha_.*$") / 60
    )

# 2022
# use left two digits of ":" and multiply by 60 and right two digits to add onto that value to go to minutes
for col in df_2022_AG1629_activities.columns:
    if not col.startswith('ha_'):
        continue

    duration_in_minutes = []

    for duration_as_str in df_2022_AG1629_activities[col]:
        hours, minutes = duration_as_str.split(':')
        duration_in_minutes.append(int(hours) * 60 + int(minutes))

    df_2022_AG1629_activities = df_2022_AG1629_activities.with_columns(
        pl.Series(col, duration_in_minutes)
    )

In [6]:
## Create a new dataframe "df_YYYY_AG1629_activites_means" with the activity codes as rows and a column for YYYY
## for each column in df_activities: create the mean and write to the row of the activity code and the column YYYY

# Create a nested list with pairs of column and mean of column: [[ha_1, mean(ha_1)], [...]] 
# and a function to calculate the weighted mean
def column_and_weighted_mean(dataframe, weight_column):
    rows = []
    weights = dataframe.get_column(weight_column)

    for column in dataframe.columns:
        if column == weight_column:
            continue

        minutes = dataframe.get_column(column)
        weighted_mean = (minutes * weights).sum() / weights.sum()

        rows.append([column, weighted_mean])

    return rows

## 1991
df_1991_AG1629_activities_means = pl.DataFrame(
    column_and_weighted_mean(
        df_1991_AG1629_activities,
        "M12",
    ),
    schema=["activity code", "mean"],
    orient="row",
)

## 2001
df_2001_AG1629_activities_means = pl.DataFrame(
    column_and_weighted_mean(
        df_2001_AG1629_activities,
        "GPZV80",
    ),
    schema=["activity code", "mean"],
    orient="row",
)

## 2012
df_2012_AG1629_activities_means = pl.DataFrame(
    column_and_weighted_mean(
        df_2012_AG1629_activities,
        "hrft80",
    ),
    schema=["activity code", "mean"],
    orient="row",
)

## 2022
df_2022_AG1629_activities_means = pl.DataFrame(
    column_and_weighted_mean(
        df_2022_AG1629_activities,
        "hrf_t80",
    ),
    schema=["activity code", "mean"],
    orient="row",
)

In [7]:
df_codes_2012 = pl.DataFrame(
    df_2012_AG1629_activities_means['activity code'],
    schema=['codes'],
    orient= 'row'
)

df_codes_2012.write_csv('2001/codes_2012.csv', separator=',')

In [8]:
## New codes for 1991 -> 2001.
df_mapping_1991_to_2001 = pl.read_csv(
    "1991/codes_1991_to_2001.csv",
    separator=','
)

## New codes for 2001 -> 2012.
df_mapping_2001_to_2012 = pl.read_csv(
    "2001/codes_2001_to_2012.csv", 
    separator=','
)

In [ ]:
## Map codes

def map_codes(df_map, df_data, mapping_year):
    code_column = df_data.columns[0]

    # two mapping columns into a lookup dictionary.
    mapping = dict(df_map.select(df_map.columns[:2]).iter_rows())

    mapped_codes = []

    for codes in df_data.get_column(code_column):
        destinations = []

        # either one code or several codes separated by "/".
        for old_code in codes.split("/"):
            if old_code.startswith("UNRESOLVED"):
                new_code = old_code
            else:
                new_code = mapping.get(old_code)

                if new_code in [None, "", "na", "UNRESOLVED"]:
                    new_code = f"UNRESOLVED:{mapping_year}:{old_code}"

            destinations.extend(new_code.split("/"))

        # Remove duplicates and keep the destinations in one row.
        unique_codes = sorted(set(destinations))
        mapped_codes.append("/".join(unique_codes))

    # Replace only the codes. All time values stay unchanged.
    return df_data.with_columns(
        pl.Series(code_column, mapped_codes)
    )

## Map 1991 to 2001
df_1991_AG1629_activities_means = map_codes(
    df_mapping_1991_to_2001, 
    df_1991_AG1629_activities_means,
    mapping_year=1991
    )

## Map 1991 to 2012
df_1991_AG1629_activities_means = map_codes(
    df_mapping_2001_to_2012, 
    df_1991_AG1629_activities_means,
    mapping_year=2001
    )

## Map 2001 to 2012
df_2001_AG1629_activities_means = map_codes(
    df_mapping_2001_to_2012, 
    df_2001_AG1629_activities_means,
    mapping_year=2001
    )

## Rename ZH... activity codes to ha_...
def rename_activity_code(value):
    if value.startswith("ZH"):
        return "ha_" + str(value[2:])
    return value

df_1991_AG1629_activities_means = df_1991_AG1629_activities_means.with_columns(
    pl.col('activity code')
    .map_elements(rename_activity_code, return_dtype=pl.String)
    .alias('activity code')
)

df_2001_AG1629_activities_means = df_2001_AG1629_activities_means.with_columns(
    pl.col('activity code')
    .map_elements(rename_activity_code, return_dtype=pl.String)
    .alias('activity code')
)

## merging rows with the same code, sum of means
df_1991_AG1629_activities_means = df_1991_AG1629_activities_means.group_by('activity code').agg(pl.col('mean').sum())
df_2001_AG1629_activities_means = df_2001_AG1629_activities_means.group_by('activity code').agg(pl.col('mean').sum())

/var/folders/21/k1wm74wx1dj0gk3xwfc4vhkh0000gn/T/ipykernel_78083/3228498498.py:32: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("_mapped_codes")


**Data set for yearly comparison**

In [10]:
## Create a new dataframe "df_Year-by-Year_AG1629_overview" with the activity codes as rows and 1991, 2001, 2012 and 2022 as columns
## Add all data points of the years from dataframes "df_YYYY_AG1629_activites" to "df_Year-by-Year_AG1629_overview"

## Create rows with the activity code, find the row in 1991, 2001, 2012 and 2022 and add their means to the row. Return the rows.

tables = [
    df_1991_AG1629_activities_means.rename({"mean": "1991"}),
    df_2001_AG1629_activities_means.rename({"mean": "2001"}),
    df_2012_AG1629_activities_means.rename({"mean": "2012"}),
    df_2022_AG1629_activities_means.rename({"mean": "2022"}),
]

df_YBY_AG1629_means = tables[0]

for table in tables[1:]:
    df_YBY_AG1629_means = df_YBY_AG1629_means.join(
        table,
        on="activity code",
        how="full",
        coalesce=True,
    )

In [11]:
## Add trends to compare year-over-year and a trend for 1991-2022 for each activity code for "df_Year-by-Year_AG1629_overview"

df_YBY_AG1629_means = df_YBY_AG1629_means.with_columns(
    (pl.col('2001') - pl.col('1991')).alias('1991 -> 2001')
)

df_YBY_AG1629_means = df_YBY_AG1629_means.with_columns(
    (pl.col('2012') - pl.col('2001')).alias('2001 -> 2012')
)

df_YBY_AG1629_means = df_YBY_AG1629_means.with_columns(
    (pl.col('2022') - pl.col('2012')).alias('2012 -> 2022')
)

df_YBY_AG1629_means = df_YBY_AG1629_means.with_columns(
    (pl.col('2022') - pl.col('1991')).alias('1991 -> 2022')
)

In [12]:
# Keep three-digit activity codes, including those ending in x
df_YBY_AG1629_means = df_YBY_AG1629_means.filter(
    pl.col("activity code").str.contains(r"^ha_[0-9]{3}x?$")
)

In [13]:
## Add label to activity codes
df_labels = pl.read_excel('2022/zve_2022_puf_dsb_v2_0.xlsx', engine='calamine', sheet_name='Summendaten')

df_labels = df_labels.drop(['Teil', 'Schlüssel'])

df_labels = df_labels.filter(
    pl.col('Variable').str.starts_with('ha_')
)

def map_labels(df_labels, df_data):
    map = dict(df_labels.iter_rows())
    activity_code = df_data.columns[0]
    return df_data.with_columns(
        pl.col(activity_code)
        .replace(map)
        .alias('label')
    )

df_YBY_AG1629_means = map_labels(
    df_labels, 
    df_YBY_AG1629_means
    )

## Clean up the data set to display the columns in the correct way and sort by the trend 1991 -> 2022
df_YBY_AG1629_means = df_YBY_AG1629_means.select('activity code', 'label', '1991', '2001', '2012', '2022', '1991 -> 2001', '2001 -> 2012', '2012 -> 2022', '1991 -> 2022').sort('1991 -> 2022')

df_YBY_AG1629_means

activity code,label,1991,2001,2012,2022,1991 -> 2001,2001 -> 2012,2012 -> 2022,1991 -> 2022
str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""ha_246""","""246 Sonstige Pausen in Haupt- …",null,null,null,0.388327,null,null,null,null
"""ha_249x""","""249x Sonstige/unbestimmte Akti…",null,null,null,0.723302,null,null,null,null
"""ha_319""","""319 Sonstige Unterrichtsfächer…",null,null,null,10.393753,null,null,null,null
"""ha_321""","""321 Angebote der Schule zur Vo…",null,null,0.065342,0.107482,null,null,0.04214,null
"""ha_329x""","""329x Sonstige/unbestimmte Betr…",null,null,null,2.252392,null,null,null,null
…,…,…,…,…,…,…,…,…,…
"""ha_411""","""411 Essen und Getränke vor- un…",8.223156,11.997832,14.779843,18.344547,3.774676,2.782011,3.564704,10.121391
"""ha_642""","""642 Familienfeiern/Feste priva…",3.079802,12.506616,14.362343,14.174171,9.426814,1.855727,-0.188172,11.094369
"""ha_120""","""120 Essen und Trinken (HA)""",68.681708,84.269651,83.006934,85.184098,15.587943,-1.262717,2.177164,16.50239


In [ ]:
## Create average and max incline/decline year over year

columns = ['1991 -> 2001', '2001 -> 2012', '2012 -> 2022']
abs_values = pl.col(columns).replace(0, None).abs()

df_YBY_AG1629_means = df_YBY_AG1629_means.with_columns(
    pl.mean_horizontal(
        abs_values
    ).alias('year-over-year (mean)')
)

df_YBY_AG1629_means = df_YBY_AG1629_means.with_columns(
    pl.max_horizontal(
        abs_values
    )
    .alias('year-over-year (max)')
)

activity code,label,1991,2001,2012,2022,1991 -> 2001,2001 -> 2012,2012 -> 2022,1991 -> 2022,year-over-year (mean),year-over-year (max)
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""ha_246""","""246 Sonstige Pausen in Haupt- …",null,null,null,0.388327,null,null,null,null,null,null
"""ha_249x""","""249x Sonstige/unbestimmte Akti…",null,null,null,0.723302,null,null,null,null,null,null
"""ha_319""","""319 Sonstige Unterrichtsfächer…",null,null,null,10.393753,null,null,null,null,null,null
"""ha_329x""","""329x Sonstige/unbestimmte Betr…",null,null,null,2.252392,null,null,null,null,null,null
"""ha_330""","""330 AGs, Schulsportteams, ande…",null,null,null,0.368046,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…
"""ha_763""","""763 Spielen am PC/Laptop und H…",2.146329,12.802559,23.816242,35.826508,10.65623,11.013683,12.010267,33.680179,11.226726,12.010267
"""ha_641""","""641 Zu Besuch sein/Besuch empf…",39.038409,13.003014,9.657474,19.213874,-26.035395,-3.34554,9.5564,-19.824535,12.979111,26.035395
"""ha_611""","""611 Kontakte/Gespräche mit Hau…",23.209387,38.719404,33.86728,6.808076,15.510018,-4.852124,-27.059204,-16.40131,15.807115,27.059204


In [17]:
df_YBY_AG1629_means.sort('year-over-year (max)')

activity code,label,1991,2001,2012,2022,1991 -> 2001,2001 -> 2012,2012 -> 2022,1991 -> 2022,year-over-year (mean),year-over-year (max)
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""ha_246""","""246 Sonstige Pausen in Haupt- …",null,null,null,0.388327,null,null,null,null,null,null
"""ha_249x""","""249x Sonstige/unbestimmte Akti…",null,null,null,0.723302,null,null,null,null,null,null
"""ha_319""","""319 Sonstige Unterrichtsfächer…",null,null,null,10.393753,null,null,null,null,null,null
"""ha_329x""","""329x Sonstige/unbestimmte Betr…",null,null,null,2.252392,null,null,null,null,null,null
"""ha_330""","""330 AGs, Schulsportteams, ande…",null,null,null,0.368046,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…
"""ha_719""","""719 Sonstiger Sport und Outdoo…",3.907804,21.025799,null,2.951437,17.117995,null,null,-0.956367,17.117995,17.117995
"""ha_369""","""369 Sonstige/unbestimmte Aktiv…",27.262836,4.806997,0.882987,0.27681,-22.45584,-3.92401,-0.606177,-26.986026,8.995342,22.45584
"""ha_641""","""641 Zu Besuch sein/Besuch empf…",39.038409,13.003014,9.657474,19.213874,-26.035395,-3.34554,9.5564,-19.824535,12.979111,26.035395


In [15]:
## Write dataframe "df_Year-by-Year_AG1629_overview" to "Year-by-Year_AG1629_overview.csv"
df_YBY_AG1629_means.write_csv('YBY_AG1629_overview.csv', separator=',')